In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet("D:/XAI_risk_score/data/home_credit_processed.parquet")

print(df.shape)
print(df.head())
print(df.dtypes)

(522596, 48)
   case_id date_decision    MONTH  WEEK_NUM  target  \
0    40626    2019-12-18  2019-12        50       0   
1    40704    2019-12-18  2019-12        50       0   
2    40734    2019-12-18  2019-12        50       0   
3    40737    2019-12-19  2019-12        50       0   
4    40766    2019-12-20  2019-12        50       0   

   maxdpdfrom6mto36m_3546853P  maxdpdlast12m_727P  maxdpdlast24m_143P  \
0                         0.0                 0.0                 0.0   
1                         0.0                 0.0                 0.0   
2                         0.0                 0.0                 0.0   
3                         0.0                 0.0                 0.0   
4                         0.0                 0.0                 0.0   

   maxdpdlast3m_392P  maxdpdlast6m_474P  ...  education_927M  \
0                0.0                0.0  ...      P97_36_170   
1                0.0                0.0  ...      P97_36_170   
2                0.0     

In [2]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nTarget distribution:\n{df['target'].value_counts()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")

Shape: (522596, 48)

Columns:
['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target', 'maxdpdfrom6mto36m_3546853P', 'maxdpdlast12m_727P', 'maxdpdlast24m_143P', 'maxdpdlast3m_392P', 'maxdpdlast6m_474P', 'numinstls_657L', 'numinstlsallpaid_934L', 'pctinstlsallpaidlat10d_839L', 'totalsettled_863A', 'totaldebt_9A', 'currdebt_22A', 'credamount_770A', 'mobilephncnt_593L', 'homephncnt_628L', 'numactivecreds_622L', 'applicationcnt_361L', 'applications30d_658L', 'applicationscnt_1086L', 'applicationscnt_464L', 'applicationscnt_629L', 'avgdbddpdlast24m_3658932P', 'amtinstpaidbefduel24m_4187115A', 'maxdbddpdlast1m_3658939P', 'maxdbddpdtollast12m_3658940P', 'maxdbddpdtollast6m_4187119P', 'numinstpaidlate1d_3546852L', 'numberofqueries_373L', 'days120_123L', 'days180_256L', 'days30_165L', 'days90_310L', 'days360_512L', 'sex_738L', 'education_927M', 'incometype_1044T', 'familystate_447L', 'empl_employedtotal_800L', 'language1_981M', 'mainoccupationinc_384A', 'age', 'maritalst_385M', 'requesttype_

In [5]:
# Tối ưu hóa dữ liệu
def reduce_mem_usage(df):
    """
    Hàm tự động tối ưu hóa kiểu dữ liệu để giảm dung lượng RAM cho Pandas DataFrame.
    Đã fix lỗi UFuncTypeError.
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Dung lượng RAM ban đầu: {start_mem:.2f} MB')

    for col in df.columns:
        # 1. Xử lý cột ngày tháng (Datetime)
        if col == 'date_decision':
            df[col] = pd.to_datetime(df[col])
            continue

        # 2. Kiểm tra chuẩn xác nếu cột là Dữ liệu số (Numeric)
        if pd.api.types.is_numeric_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()

            # Nếu là số nguyên (Integer)
            if pd.api.types.is_integer_dtype(df[col]):
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)

            # Nếu là số thực (Float)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

        # 3. Xử lý cột chuỗi văn bản / Phân loại (Object/String)
        else:
            num_unique_values = len(df[col].unique())
            num_total_values = len(df[col])
            if num_unique_values / num_total_values < 0.5:
                df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Dung lượng RAM sau khi tối ưu: {end_mem:.2f} MB')
    print(f'Giảm được: {100 * (start_mem - end_mem) / start_mem:.1f}%')

    return df

# Chạy lại hàm trên tập dữ liệu của bạn
df = reduce_mem_usage(df)

Dung lượng RAM ban đầu: 224.69 MB
Dung lượng RAM sau khi tối ưu: 79.74 MB
Giảm được: 64.5%


ĐIỀU LẠI KIỂU DỮ LIỆU CHO TARGET, ĐÁNH CATEGORY CHO CÁC CỘT OBJECT, ĐỊNH DẠNG LẠI KIỂU DỮ LIỆU NGÀY THÁNG

In [6]:
# Kiểm tra dữ liệu thiếu
# 1: dữ liệu về độ trễ hạn
missing_dpd_mask = df['maxdpdfrom6mto36m_3546853P'].isnull()
df_missing_dpd = df[missing_dpd_mask]

# 2. KIỂM CHỨNG 1: Có đúng là chúng khuyết thiếu cùng nhau theo một khối (Block)?
dpd_columns = [
    'maxdpdfrom6mto36m_3546853P', 'maxdpdlast12m_727P',
    'maxdpdlast24m_143P', 'maxdpdlast3m_392P', 'maxdpdlast6m_474P'
]

print("--- KIỂM TRA MỨC ĐỘ KHUYẾT ĐỒNG THỜI ---")
print(df_missing_dpd[dpd_columns].isnull().sum())
# Nếu tất cả đều in ra đúng 83798, giả thuyết "khuyết theo khối" là ĐÚNG.

# 3. KIỂM CHỨNG 2: Nhóm này có thực sự là "Khách hàng mới" không?
# Nếu là khách hàng mới, họ không thể có lịch sử trả nợ (số tiền đã tất toán) hoặc hợp đồng cũ.
credit_history_cols = [
    'totalsettled_863A',    # Tổng số tiền đã thanh toán trong quá khứ
    'numinstls_657L',       # Tổng số kỳ hạn thanh toán
    'numactivecreds_622L'   # Số lượng khoản vay đang hoạt động
]

print("\n--- THỐNG KÊ LỊCH SỬ TÍN DỤNG CỦA NHÓM BỊ THIẾU DPD ---")
print(df_missing_dpd[credit_history_cols].describe())

--- KIỂM TRA MỨC ĐỘ KHUYẾT ĐỒNG THỜI ---
maxdpdfrom6mto36m_3546853P    83798
maxdpdlast12m_727P            83798
maxdpdlast24m_143P            83798
maxdpdlast3m_392P             83798
maxdpdlast6m_474P             83798
dtype: int64

--- THỐNG KÊ LỊCH SỬ TÍN DỤNG CỦA NHÓM BỊ THIẾU DPD ---
       totalsettled_863A  numinstls_657L  numactivecreds_622L
count       83798.000000         83798.0              83798.0
mean            0.027447             0.0                  0.0
std             3.768320             0.0                  0.0
min             0.000000             0.0                  0.0
25%             0.000000             0.0                  0.0
50%             0.000000             0.0                  0.0
75%             0.000000             0.0                  0.0
max           700.000000             0.0                  0.0


In [8]:
import os
dpd_columns = [
    'maxdpdfrom6mto36m_3546853P', 'maxdpdlast12m_727P',
    'maxdpdlast24m_143P', 'maxdpdlast3m_392P', 'maxdpdlast6m_474P'
]

print("Bắt đầu xử lý dữ liệu khuyết thiếu...")

# Bước 1: Tạo Biến cờ (Shadow Variable)
df['is_new_customer'] = df['maxdpdfrom6mto36m_3546853P'].isna().astype(np.int8)

# Bước 2: Điền số 0 vào cột gốc
df[dpd_columns] = df[dpd_columns].fillna(0)

# Bước 3: Thiết lập đường dẫn lưu file theo yêu cầu
output_dir = r"D:\XAI_risk_score\data"
output_path = os.path.join(output_dir, "credit_data_processed_v1.parquet")

# Kiểm tra và tự động tạo thư mục nếu nó chưa tồn tại trên máy của bạn
os.makedirs(output_dir, exist_ok=True)

# Bước 4: Lưu toàn bộ bảng dữ liệu ra file Parquet mới
df.to_parquet(output_path, engine='pyarrow', index=False)

print(f"Đã lưu thành công tập dữ liệu với kích thước {df.shape} ra file: {output_path}")

Bắt đầu xử lý dữ liệu khuyết thiếu...
Đã lưu thành công tập dữ liệu với kích thước (522596, 49) ra file: D:\XAI_risk_score\data\credit_data_processed_v1.parquet


In [10]:
# ---------------------------------------------------------
# KỊCH BẢN 1: Kiểm chứng "Cú sập API" (Khối 20.869 dòng)
# ---------------------------------------------------------
bureau_cols = [
    'numberofqueries_373L', 'days120_123L', 'days180_256L',
    'days30_165L', 'days90_310L', 'days360_512L'
]

# Tạo DataFrame chỉ chứa cờ True/False (True nếu bị khuyết)
missing_bureau = df[bureau_cols].isnull()

print("--- 1. MA TRẬN TƯƠNG QUAN KHUYẾT THIẾU (NHÓM BUREAU / API) ---")
# Tính ma trận tương quan trên các giá trị boolean
print(missing_bureau.corr())
print("\n")

# ---------------------------------------------------------
# KỊCH BẢN 2: Kiểm chứng "Form đăng ký UI/UX"
# ---------------------------------------------------------
form_cols = ['familystate_447L', 'empl_employedtotal_800L', 'requesttype_4525192L']

missing_form = df[form_cols].isnull()

print("--- 2. MA TRẬN TƯƠNG QUAN KHUYẾT THIẾU (NHÓM NHÂN KHẨU HỌC) ---")
print(missing_form.corr())
print("\n")

# ---------------------------------------------------------
# KỊCH BẢN 3: Đếm số lượng kịch bản thiếu chéo (Cross-Tabulation)
# ---------------------------------------------------------
# Khách hàng không điền tình trạng gia đình thì có điền thời gian làm việc không?
cross_check = pd.crosstab(
    df['familystate_447L'].isnull(),
    df['empl_employedtotal_800L'].isnull(),
    rownames=['Thiếu_Family'],
    colnames=['Thiếu_Employment']
)
print("--- 3. PHÂN BỐ LỖI NGƯỜI DÙNG BỎ TRỐNG FORM ---")
print(cross_check)

--- 1. MA TRẬN TƯƠNG QUAN KHUYẾT THIẾU (NHÓM BUREAU / API) ---
                      numberofqueries_373L  days120_123L  days180_256L  \
numberofqueries_373L                   1.0           1.0           1.0   
days120_123L                           1.0           1.0           1.0   
days180_256L                           1.0           1.0           1.0   
days30_165L                            1.0           1.0           1.0   
days90_310L                            1.0           1.0           1.0   
days360_512L                           1.0           1.0           1.0   

                      days30_165L  days90_310L  days360_512L  
numberofqueries_373L          1.0          1.0           1.0  
days120_123L                  1.0          1.0           1.0  
days180_256L                  1.0          1.0           1.0  
days30_165L                   1.0          1.0           1.0  
days90_310L                   1.0          1.0           1.0  
days360_512L                  1.0       

In [14]:
# Thiết lập đường dẫn
input_path = r"D:\XAI_risk_score\data\credit_data_processed_v1.parquet"
output_path = r"D:\XAI_risk_score\data\credit_data_processed_v2.parquet"

print(f"Đang đọc dữ liệu từ: {input_path}")
df = pd.read_parquet(input_path, engine='pyarrow')

print("Tiến hành dọn dẹp các giá trị khuyết thiếu còn lại...")

# ---------------------------------------------------------
# NHÓM 1: Xử lý biến Categorical -> Điền 'UNKNOWN'
# ---------------------------------------------------------
cat_missing_cols = ['familystate_447L', 'empl_employedtotal_800L', 'requesttype_4525192L']

for col in cat_missing_cols:
    # Nếu cột đã được ép về kiểu 'category' từ bước giảm RAM
    if df[col].dtype.name == 'category':
        # Phải thêm 'UNKNOWN' vào danh sách categories cho phép trước khi fillna
        if 'UNKNOWN' not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories('UNKNOWN')

    df[col] = df[col].fillna('UNKNOWN')


# ---------------------------------------------------------
# NHÓM 2: Nhóm lỗi API truy vấn (20.869 dòng) -> Điền 0
# ---------------------------------------------------------
bureau_cols = [
    'numberofqueries_373L', 'days120_123L', 'days180_256L',
    'days30_165L', 'days90_310L', 'days360_512L'
]
df[bureau_cols] = df[bureau_cols].fillna(0)


# ---------------------------------------------------------
# NHÓM 4: Lỗi lặt vặt (1-2 dòng) -> Điền 0
# ---------------------------------------------------------
tiny_missing_cols = ['numinstls_657L', 'totalsettled_863A', 'totaldebt_9A', 'currdebt_22A']
df[tiny_missing_cols] = df[tiny_missing_cols].fillna(0)


# ---------------------------------------------------------
# NHÓM 3.1: Số lượng và Khối lượng thanh toán (Counts & Amounts) -> Điền 0
# Ý nghĩa: Không có hợp đồng đang chạy = 0 kỳ được trả, 0 VND được đóng.
# ---------------------------------------------------------
count_amount_cols = [
    'numinstlsallpaid_934L',
    'numinstpaidlate1d_3546852L',
    'amtinstpaidbefduel24m_4187115A'
]
df[count_amount_cols] = df[count_amount_cols].fillna(0)


# ---------------------------------------------------------
# NHÓM 3.2: Tỷ lệ, Trung bình và Max (Ratios, Averages, Max) -> Điền -1
# Ý nghĩa: Tách biệt "Trả nợ xuất sắc (0)" và "Không có giao dịch để tính toán (-1)"
# ---------------------------------------------------------
ratio_avg_max_cols = [
    'pctinstlsallpaidlat10d_839L',
    'avgdbddpdlast24m_3658932P',
    'maxdbddpdlast1m_3658939P',
    'maxdbddpdtollast12m_3658940P',
    'maxdbddpdtollast6m_4187119P'
]
df[ratio_avg_max_cols] = df[ratio_avg_max_cols].fillna(-1)

# (Tùy chọn) Chốt chặn an toàn: Quét lại xem còn sót cột numeric nào do khai báo thiếu không
missed_numeric_cols = df.select_dtypes(include=[np.number]).columns[df.select_dtypes(include=[np.number]).isnull().any()].tolist()
if missed_numeric_cols:
    print(f"Cảnh báo: Còn sót các cột chưa định danh: {missed_numeric_cols}")
    df[missed_numeric_cols] = df[missed_numeric_cols].fillna(-1)

# ---------------------------------------------------------
# LƯU FILE PARQUET CUỐI CÙNG
# ---------------------------------------------------------
df.to_parquet(output_path, engine='pyarrow', index=False)

print(f"Đã lưu tập dữ liệu sạch hoàn toàn ({df.shape[0]} dòng, {df.shape[1]} cột) ra file: {output_path}")

# Kiểm tra chốt chặn cuối cùng
total_nulls = df.isnull().sum().sum()
print(f"Tổng số lượng giá trị NULL còn sót lại trong toàn bộ bảng: {total_nulls}")

Đang đọc dữ liệu từ: D:\XAI_risk_score\data\credit_data_processed_v1.parquet
Tiến hành dọn dẹp các giá trị khuyết thiếu còn lại...
Đã lưu tập dữ liệu sạch hoàn toàn (522596 dòng, 49 cột) ra file: D:\XAI_risk_score\data\credit_data_processed_v2.parquet
Tổng số lượng giá trị NULL còn sót lại trong toàn bộ bảng: 0


In [17]:
df1 = pd.read_parquet("D:/XAI_risk_score/data/credit_data_processed_v2.parquet")
print(f"Shape: {df1.shape}")
print(f"\nColumns:\n{df1.columns.tolist()}")
print(f"\nTarget distribution:\n{df1['target'].value_counts()}")
print(f"\nMissing values:\n{df1.isnull().sum()}")
print(f"\nData types:\n{df1.dtypes}")

Shape: (522596, 49)

Columns:
['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target', 'maxdpdfrom6mto36m_3546853P', 'maxdpdlast12m_727P', 'maxdpdlast24m_143P', 'maxdpdlast3m_392P', 'maxdpdlast6m_474P', 'numinstls_657L', 'numinstlsallpaid_934L', 'pctinstlsallpaidlat10d_839L', 'totalsettled_863A', 'totaldebt_9A', 'currdebt_22A', 'credamount_770A', 'mobilephncnt_593L', 'homephncnt_628L', 'numactivecreds_622L', 'applicationcnt_361L', 'applications30d_658L', 'applicationscnt_1086L', 'applicationscnt_464L', 'applicationscnt_629L', 'avgdbddpdlast24m_3658932P', 'amtinstpaidbefduel24m_4187115A', 'maxdbddpdlast1m_3658939P', 'maxdbddpdtollast12m_3658940P', 'maxdbddpdtollast6m_4187119P', 'numinstpaidlate1d_3546852L', 'numberofqueries_373L', 'days120_123L', 'days180_256L', 'days30_165L', 'days90_310L', 'days360_512L', 'sex_738L', 'education_927M', 'incometype_1044T', 'familystate_447L', 'empl_employedtotal_800L', 'language1_981M', 'mainoccupationinc_384A', 'age', 'maritalst_385M', 'requesttype_

In [18]:
# Hàm phân tích Bad Rate tự động cho các biến phân loại (Categorical)
def analyze_bad_rate(df, column_name):
    print(f"\n--- PHÂN TÍCH RỦI RO THEO: {column_name.upper()} ---")

    # Gom nhóm theo cột và tính toán
    summary = df.groupby(column_name, observed=False).agg(
        Total_Customers=('target', 'count'),
        Bad_Customers=('target', 'sum')
    )

    # Tính tỷ lệ % nợ xấu (Bad Rate)
    summary['Bad_Rate_%'] = (summary['Bad_Customers'] / summary['Total_Customers']) * 100

    # Sắp xếp từ rủi ro cao nhất xuống thấp nhất
    summary = summary.sort_values(by='Bad_Rate_%', ascending=False)

    # Lọc bỏ những nhóm có quá ít khách hàng (nhiễu thống kê)
    summary = summary[summary['Total_Customers'] > 100]

    # Format hiển thị
    summary['Bad_Rate_%'] = summary['Bad_Rate_%'].round(2)
    print(summary)

# Chạy thử với 3 cột nhân khẩu học quan trọng
columns_to_check = ['education_927M', 'incometype_1044T', 'familystate_447L']

for col in columns_to_check:
    analyze_bad_rate(df, col)


--- PHÂN TÍCH RỦI RO THEO: EDUCATION_927M ---
                Total_Customers  Bad_Customers  Bad_Rate_%
education_927M                                            
P157_18_172                 216             17        7.87
P97_36_170               116442           5223        4.49
P17_36_170                 1859             83        4.46
a55475b1                 279907           8700        3.11
P33_146_175               74049           1985        2.68
P106_81_188               50123           1216        2.43

--- PHÂN TÍCH RỦI RO THEO: INCOMETYPE_1044T ---
                         Total_Customers  Bad_Customers  Bad_Rate_%
incometype_1044T                                                   
PRIVATE_SECTOR_EMPLOYEE           139410           6195        4.44
EMPLOYED                          100050           4104        4.10
HANDICAPPED_2                       1849             71        3.84
HANDICAPPED_3                       1463             47        3.21
SALARIED_GOVT           